In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction import FeatureHasher
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from joblib import dump, load
import os

In [2]:
warnings.filterwarnings("ignore")

class ChampionHasher(BaseEstimator, TransformerMixin):
    def __init__(self, n_features=200):
        self.n_features = n_features
        self.hasher = None

    def fit(self, X, y=None):
        self.hasher = FeatureHasher(n_features=self.n_features, input_type='string')
        return self

    def transform(self, X):
        def hash_team(picks):
            return self.hasher.transform([[c.strip().title() for c in picks]]).toarray()[0]
        blue = np.vstack(X['blue_picks'].apply(hash_team))
        red = np.vstack(X['red_picks'].apply(hash_team))
        return np.hstack([blue, red])

class TeamTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, label):
        self.label = label
        self.team_means = {}

    def fit(self, X, y):
        df = X.copy()
        df[self.label] = y
        means = {}
        for col in ['blue_team', 'red_team']:
            means[col] = df.groupby(col)[self.label].mean().to_dict()
        self.team_means = means
        return self

    def transform(self, X):
        X_enc = pd.DataFrame()
        X_enc['blue_team_enc'] = X['blue_team'].map(self.team_means['blue_team']).fillna(0.5)
        X_enc['red_team_enc'] = X['red_team'].map(self.team_means['red_team']).fillna(0.5)
        return X_enc.to_numpy()

class KillsOverUnderPredictor:
    def __init__(self):
        self.models = {}
        self.thresholds = [x + 0.5 for x in range(20, 35)]
        self.model_dir = os.path.join("models", "kill_predictors")
        os.makedirs(self.model_dir, exist_ok=True)

    def load_data(self, path):
        df = pd.read_csv(path, usecols=[
            'gameid', 'side', 'champion', 'teamname', 'result', 'date',
            'participantid', 'league', 'kills'
        ])
        df['kills'] = pd.to_numeric(df['kills'], errors='coerce')

        matches = []
        for game_id in df['gameid'].unique():
            g = df[df['gameid'] == game_id]
            players = g[g['champion'].notna() & g['side'].isin(['Blue', 'Red'])]
            teams = g[g['champion'].isna() & g['side'].isin(['Blue', 'Red'])]

            if len(teams) != 2 or len(players) != 10:
                continue

            try:
                blue_picks = players[players['side'].str.lower() == 'blue']['champion'].tolist()
                red_picks = players[players['side'].str.lower() == 'red']['champion'].tolist()

                if len(blue_picks) != 5 or len(red_picks) != 5:
                    continue

                blue_team = teams[teams['side'].str.lower() == 'blue'].iloc[0]
                red_team = teams[teams['side'].str.lower() == 'red'].iloc[0]

                bk, rk = blue_team['kills'], red_team['kills']
                if pd.isna(bk) or pd.isna(rk):
                    continue

                total = int(bk) + int(rk)
                match = {
                    'blue_picks': blue_picks,
                    'red_picks': red_picks,
                    'blue_team': blue_team['teamname'],
                    'red_team': red_team['teamname']
                }
                for t in self.thresholds:
                    match[f'over_{str(t).replace(".", "_")}'] = int(total > t)
                matches.append(match)

            except Exception:
                continue

        print(f"Parsed {len(matches)} matches with over/under kill labels.")
        return pd.DataFrame(matches)

    def train_single(self, X, y, label):
        model_path = os.path.join(self.model_dir, f"{label}.joblib")

        pipeline = Pipeline([
            ('features', ColumnTransformer([
                ('champions', ChampionHasher(n_features=400), ['blue_picks', 'red_picks']),
                ('teams', TeamTargetEncoder(label=label), ['blue_team', 'red_team']),
            ])),
            ('classifier', XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=42
            ))
        ])

        X_train, X_test, y_train, y_test = train_test_split(X, y[label], test_size=0.2, shuffle=False)
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"{label}: Accuracy = {acc:.2%}")

        dump(pipeline, model_path)
        self.models[label] = pipeline

    def train(self, path):
        df = self.load_data(path)
        if df.empty:
            print("No data to train on.")
            return
        X = df[['blue_picks', 'red_picks', 'blue_team', 'red_team']]
        for t in self.thresholds:
            label = f'over_{str(t).replace(".", "_")}'
            self.train_single(X, df, label)

    def load(self):
        for t in self.thresholds:
            label = f'over_{str(t).replace(".", "_")}'
            model_path = os.path.join(self.model_dir, f"{label}.joblib")
            if os.path.exists(model_path):
                self.models[label] = load(model_path)
            else:
                print(f"Model not found: {model_path}")
        return self

    def predict(self, blue_picks, red_picks, blue_team="Unknown", red_team="Unknown"):
        input_df = pd.DataFrame([{
            'blue_picks': [c.strip().title() for c in blue_picks],
            'red_picks': [c.strip().title() for c in red_picks],
            'blue_team': blue_team,
            'red_team': red_team
        }])

        results = {}
        for label, model in self.models.items():
            proba = model.predict_proba(input_df)[0][1]
            results[label] = round(proba, 3)

        return results

In [3]:
predictor = KillsOverUnderPredictor()
predictor.train('matches.csv')  # Only run once; skip if already trained
predictor.load()

result = predictor.predict(
    blue_picks=["Darius", "Nidalee", "Jayce", "Ezreal", "Alistar"],
    red_picks=["Ambessa", "Skarner", "Aurora", "Varus", "Rakan"],
    blue_team="BoostGate Esports",
    red_team="BBL Dark Passage"
)

print("Probabilities for Over Kill Totals:")
for k, v in sorted(result.items(), key=lambda x: float(x[0].split("_")[1].replace("_", "."))):
    print(f"{k.replace('_', '.')} kills: {v:.1%}")

Parsed 6173 matches with over/under kill labels.
over_20_5: Accuracy = 86.64%
over_21_5: Accuracy = 83.89%
over_22_5: Accuracy = 80.16%
over_23_5: Accuracy = 79.11%
over_24_5: Accuracy = 74.01%
over_25_5: Accuracy = 69.23%
over_26_5: Accuracy = 68.10%
over_27_5: Accuracy = 67.13%
over_28_5: Accuracy = 64.70%
over_29_5: Accuracy = 63.16%
over_30_5: Accuracy = 61.78%
over_31_5: Accuracy = 59.60%
over_32_5: Accuracy = 61.94%
over_33_5: Accuracy = 62.51%
over_34_5: Accuracy = 63.40%
Probabilities for Over Kill Totals:
over.20.5 kills: 90.3%
over.21.5 kills: 89.1%
over.22.5 kills: 88.1%
over.23.5 kills: 84.3%
over.24.5 kills: 83.4%
over.25.5 kills: 72.4%
over.26.5 kills: 66.5%
over.27.5 kills: 61.1%
over.28.5 kills: 60.8%
over.29.5 kills: 49.9%
over.30.5 kills: 20.1%
over.31.5 kills: 20.4%
over.32.5 kills: 25.9%
over.33.5 kills: 27.7%
over.34.5 kills: 22.8%
